# 03. 주문 단위 마스터 테이블 (Day 3)
**목적**: 아이템 단위 표를 주문 1행으로 접고(대표 아이템 규칙), 리뷰를 LEFT JOIN해서 모델링용 마스터 테이블을 만든다.

In [1]:
# 참고: item_base, rep_price, rep_dist, ord_sum, ord_pay, ord_agg, master는 아래 셀에서 만드는 테이블이라 재실행하면 목록에 함께 보임
import duckdb

con = duckdb.connect("data/interim/olist.duckdb")
con.execute("SHOW TABLES").df().name.tolist()

['base',
 'cat_tr',
 'chk',
 'customers',
 'geo_clean',
 'geo_rep',
 'geolocation',
 'item_base',
 'item_lvl',
 'master',
 'ord_agg',
 'ord_pay',
 'ord_sum',
 'order_items',
 'order_payments',
 'order_reviews',
 'orders',
 'products',
 'rep_dist',
 'rep_price',
 'rev',
 'sellers',
 'zip_status']

In [2]:
con.execute("""
SELECT 'base' AS tbl, COUNT(*) AS n_rows, COUNT(DISTINCT order_id) AS n_orders FROM base
UNION ALL
SELECT 'item_lvl', COUNT(*), COUNT(DISTINCT order_id) FROM item_lvl
UNION ALL
SELECT 'rev', COUNT(*), COUNT(DISTINCT order_id) FROM rev
UNION ALL
SELECT 'order_payments', COUNT(*), COUNT(DISTINCT order_id) FROM order_payments
""").df()

,tbl,n_rows,n_orders
0,base,96204,96204
1,item_lvl,112650,98666
2,rev,98673,98673
3,order_payments,103886,99440


In [3]:
con.execute("""
CREATE OR REPLACE TABLE item_base AS
SELECT i.order_id, i.order_item_id, i.product_id, i.seller_id, i.price, i.freight_value,
       i.distance_km, i.c_n, i.s_n,
       p.product_category_name AS category_pt,
       COALESCE(t.product_category_name_english, '미상') AS category_en
FROM item_lvl i
JOIN base b ON b.order_id = i.order_id
LEFT JOIN products p ON p.product_id = i.product_id
LEFT JOIN cat_tr t ON t.product_category_name = p.product_category_name
""")

display(con.execute("""
SELECT (SELECT COUNT(*) FROM item_lvl WHERE order_id IN (SELECT order_id FROM base)) AS expected_rows,
       COUNT(*) AS n_item_base, COUNT(DISTINCT order_id) AS n_orders,
       CAST(SUM(CASE WHEN category_en = '미상' THEN 1 ELSE 0 END) AS INTEGER) AS n_unknown_cat,
       CAST(SUM(CASE WHEN category_pt IS NULL THEN 1 ELSE 0 END) AS INTEGER) AS n_cat_null,
       CAST(SUM(CASE WHEN category_pt IS NOT NULL AND category_en = '미상' THEN 1 ELSE 0 END) AS INTEGER) AS n_untranslated
FROM item_base
""").df())

display(con.execute("SELECT COUNT(*) AS n_products, COUNT(DISTINCT product_id) AS n_unique FROM products").df())

,expected_rows,n_item_base,n_orders,n_unknown_cat,n_cat_null,n_untranslated
0,109873,109873,96204,1557,1535,22


,n_products,n_unique
0,32951,32951


In [4]:
# [셀 4] 대표 아이템 2종 → 기대: rep_price 96,204 / 96,204, rep_dist 96,204 / 96,204
con.execute("""
CREATE OR REPLACE TABLE rep_price AS
SELECT order_id, order_item_id AS item_by_price, seller_id AS seller_by_price, category_en, category_pt
FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY price DESC, order_item_id ASC) AS rn
      FROM item_base)
WHERE rn = 1
""")

con.execute("""
CREATE OR REPLACE TABLE rep_dist AS
SELECT order_id, order_item_id AS item_by_dist, seller_id AS seller_rep, distance_km
FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY distance_km DESC NULLS LAST, order_item_id ASC) AS rn
      FROM item_base)
WHERE rn = 1
""")

con.execute("""
SELECT 'rep_price' AS tbl, COUNT(*) AS n_rows, COUNT(DISTINCT order_id) AS n_orders FROM rep_price
UNION ALL
SELECT 'rep_dist', COUNT(*), COUNT(DISTINCT order_id) FROM rep_dist
""").df()

,tbl,n_rows,n_orders
0,rep_price,96204,96204
1,rep_dist,96204,96204


In [5]:
# [셀 5] 합산·결제·ord_agg → 기대: ord_sum 96,204 / ord_pay 99,440 / ord_agg 96,204(유일)
#                              멀티셀러 1,272 / 결제 없는 주문 0
con.execute("""
CREATE OR REPLACE TABLE ord_sum AS
SELECT order_id, SUM(price) AS price_sum, SUM(freight_value) AS freight_sum,
       COUNT(*) AS n_items, COUNT(DISTINCT seller_id) AS n_sellers
FROM item_base
GROUP BY order_id
""")

con.execute("""
CREATE OR REPLACE TABLE ord_pay AS
SELECT order_id, MAX(payment_installments) AS installments_max
FROM order_payments
GROUP BY order_id
""")

con.execute("""
CREATE OR REPLACE TABLE ord_agg AS
SELECT s.order_id, s.price_sum, s.freight_sum, s.n_items, s.n_sellers, p.installments_max,
       rp.category_en, rp.category_pt, rp.seller_by_price, rp.item_by_price,
       rd.seller_rep, rd.item_by_dist, rd.distance_km
FROM ord_sum s
JOIN rep_price rp ON rp.order_id = s.order_id
JOIN rep_dist  rd ON rd.order_id = s.order_id
LEFT JOIN ord_pay p ON p.order_id = s.order_id
""")

display(con.execute("""
SELECT 'ord_sum' AS tbl, COUNT(*) AS n_rows, COUNT(DISTINCT order_id) AS n_orders FROM ord_sum
UNION ALL SELECT 'ord_pay', COUNT(*), COUNT(DISTINCT order_id) FROM ord_pay
UNION ALL SELECT 'ord_agg', COUNT(*), COUNT(DISTINCT order_id) FROM ord_agg
""").df())

display(con.execute("""
SELECT COUNT(*) AS n_orders,
       CAST(SUM(CASE WHEN n_sellers > 1 THEN 1 ELSE 0 END) AS INTEGER) AS n_multi_seller,
       CAST(SUM(CASE WHEN installments_max IS NULL THEN 1 ELSE 0 END) AS INTEGER) AS n_no_payment
FROM ord_agg
""").df())

,tbl,n_rows,n_orders
0,ord_sum,96204,96204
1,ord_pay,99440,99440
2,ord_agg,96204,96204


,n_orders,n_multi_seller,n_no_payment
0,96204,1272,0


In [6]:
# [셀 6] D-05 → 기대: 1,272 / 셀러 기준 623(49.0%) / 아이템 기준 615(48.3%) / 다른 아이템·같은 셀러 8
display(con.execute("""
SELECT COUNT(*) AS n_multi,
       CAST(SUM(CASE WHEN seller_rep = seller_by_price THEN 1 ELSE 0 END) AS INTEGER) AS same_seller,
       ROUND(100.0 * AVG(CASE WHEN seller_rep = seller_by_price THEN 1 ELSE 0 END), 1) AS pct_same_seller,
       CAST(SUM(CASE WHEN item_by_dist = item_by_price THEN 1 ELSE 0 END) AS INTEGER) AS same_item,
       ROUND(100.0 * AVG(CASE WHEN item_by_dist = item_by_price THEN 1 ELSE 0 END), 1) AS pct_same_item,
       CAST(SUM(CASE WHEN item_by_dist <> item_by_price AND seller_rep = seller_by_price THEN 1 ELSE 0 END) AS INTEGER) AS diff_item_same_seller
FROM ord_agg
WHERE n_sellers > 1
""").df())

,n_multi,same_seller,pct_same_seller,same_item,pct_same_item,diff_item_same_seller
0,1272,623,49.0,615,48.3,8


In [7]:
# [셀 7] D-12 근거 → 기대: 96,204 / 전부 NULL 477(그중 멀티셀러 6) / 일부만 NULL 4(전부 멀티셀러 4)
display(con.execute("""
SELECT COUNT(*) AS n_orders,
       CAST(SUM(CASE WHEN n_null = n_items THEN 1 ELSE 0 END) AS INTEGER) AS all_null,
       CAST(SUM(CASE WHEN n_null = n_items AND n_sellers > 1 THEN 1 ELSE 0 END) AS INTEGER) AS all_null_multi_seller,
       CAST(SUM(CASE WHEN n_null > 0 AND n_null < n_items THEN 1 ELSE 0 END) AS INTEGER) AS partial_null,
       CAST(SUM(CASE WHEN n_null > 0 AND n_null < n_items AND n_sellers > 1 THEN 1 ELSE 0 END) AS INTEGER) AS partial_null_multi_seller
FROM (SELECT order_id, COUNT(*) AS n_items, COUNT(DISTINCT seller_id) AS n_sellers,
             SUM(CASE WHEN distance_km IS NULL THEN 1 ELSE 0 END) AS n_null
      FROM item_base GROUP BY order_id)
""").df())

,n_orders,all_null,all_null_multi_seller,partial_null,partial_null_multi_seller
0,96204,477,6,4,4


In [8]:
# [셀 8] D-13 동점 민감도 → 기대: 동점 7,431주문(카테고리 다른 동점 56) / 미상 1,376 vs 1,379 / 카테고리 변경 53 / 미상 여부 변경 9
display(con.execute("""
WITH m AS (SELECT order_id, MAX(price) AS mx FROM item_base GROUP BY order_id),
t AS (SELECT i.order_id, COUNT(*) AS n_top, COUNT(DISTINCT i.category_en) AS n_cat_top
      FROM item_base i JOIN m ON m.order_id = i.order_id AND i.price = m.mx
      GROUP BY i.order_id)
SELECT COUNT(*) AS n_orders,
       CAST(SUM(CASE WHEN n_top > 1 THEN 1 ELSE 0 END) AS INTEGER) AS price_tie_orders,
       CAST(SUM(CASE WHEN n_top > 1 AND n_cat_top > 1 THEN 1 ELSE 0 END) AS INTEGER) AS tie_with_diff_category
FROM t
""").df())

display(con.execute("""
WITH a AS (SELECT order_id, category_en FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY price DESC, order_item_id ASC) AS rn FROM item_base) WHERE rn = 1),
d AS (SELECT order_id, category_en FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY price DESC, order_item_id DESC) AS rn FROM item_base) WHERE rn = 1)
SELECT (SELECT COUNT(*) FROM a WHERE category_en = '미상') AS unknown_asc,
       (SELECT COUNT(*) FROM d WHERE category_en = '미상') AS unknown_desc,
       (SELECT COUNT(*) FROM a JOIN d USING (order_id) WHERE a.category_en <> d.category_en) AS category_differs,
       (SELECT COUNT(*) FROM a JOIN d USING (order_id) WHERE (a.category_en = '미상') <> (d.category_en = '미상')) AS unknown_flag_differs
""").df())

,n_orders,price_tie_orders,tie_with_diff_category
0,96204,7431,56


,unknown_asc,unknown_desc,category_differs,unknown_flag_differs
0,1376,1379,53,9


In [9]:
# [셀 9] master 생성 → 기대: 96,204 / 96,204, 컬럼 26개
con.execute("""
CREATE OR REPLACE TABLE master AS
SELECT b.order_id, b.customer_id, b.order_status,
       b.order_purchase_timestamp AS purchase_ts,
       MONTH(b.order_purchase_timestamp) AS purchase_month,
       b.order_approved_at, b.order_delivered_customer_date, b.order_estimated_delivery_date,
       b.late,
       c.customer_state,
       sl.seller_state,
       a.price_sum, a.freight_sum, a.n_items, a.n_sellers, a.installments_max,
       a.category_en, a.category_pt, a.seller_by_price, a.seller_rep,
       a.distance_km, CASE WHEN a.distance_km IS NULL THEN 1 ELSE 0 END AS dist_missing,
       r.review_id, r.review_score,
       CASE WHEN r.review_score IS NULL THEN NULL
            WHEN r.review_score <= 2 THEN 1 ELSE 0 END AS low,
       r.review_comment_message
FROM base b
JOIN customers c  ON c.customer_id = b.customer_id
JOIN ord_agg   a  ON a.order_id    = b.order_id
JOIN sellers   sl ON sl.seller_id  = a.seller_rep
LEFT JOIN rev  r  ON r.order_id    = b.order_id
""")

display(con.execute("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT order_id) AS n_orders FROM master").df())
print("컬럼 수:", len(con.execute("DESCRIBE master").df()))

,n_rows,n_orders
0,96204,96204


컬럼 수: 26


In [10]:
# [셀 10] 요약·D-06·흐름표
# 기대: 지연 6,532(6.79) / 리뷰 있음 95,561·없음 643 / 저평점 12,228(12.8, 분모=리뷰 있는 95,561) / 거리 없음 477 / 미상 1,376
display(con.execute("""
SELECT CAST(SUM(late) AS INTEGER) AS n_late, ROUND(100.0 * AVG(late), 2) AS late_pct,
       COUNT(review_score) AS n_with_review, COUNT(*) - COUNT(review_score) AS n_no_review,
       CAST(SUM(low) AS INTEGER) AS n_low, ROUND(100.0 * AVG(low), 1) AS low_pct_of_reviewed,
       CAST(SUM(dist_missing) AS INTEGER) AS n_dist_missing,
       CAST(SUM(CASE WHEN category_en = '미상' THEN 1 ELSE 0 END) AS INTEGER) AS n_unknown_cat
FROM master
""").df())

# 3단계 대상 → 기대: 8,247 / 6,449 (Day 1 값과 같아야 함)
display(con.execute("""
SELECT COUNT(*) AS n_stage3, COUNT(review_comment_message) AS n_with_comment
FROM master WHERE late = 0 AND low = 1
""").df())

# D-06: 결제 회차 → 기대: 0 / 24 / 0값 2건
display(con.execute("""
SELECT MIN(installments_max) AS min_inst, MAX(installments_max) AS max_inst,
       CAST(SUM(CASE WHEN installments_max = 0 THEN 1 ELSE 0 END) AS INTEGER) AS n_zero
FROM master
""").df())
display(con.execute("SELECT order_id, installments_max, price_sum FROM master WHERE installments_max = 0").df())

# 행 수 흐름 → 기대: 99,441 / 99,092 / 96,204 / 96,204 / 95,561
display(con.execute("""
SELECT (SELECT COUNT(*) FROM orders) AS orders_all,
       (SELECT COUNT(*) FROM orders WHERE order_purchase_timestamp >= '2017-01-01'
                                     AND order_purchase_timestamp <  '2018-09-01') AS in_window,
       (SELECT COUNT(*) FROM base) AS base,
       COUNT(*) AS master,
       COUNT(review_score) AS master_with_review
FROM master
""").df())

,n_late,late_pct,n_with_review,n_no_review,n_low,low_pct_of_reviewed,n_dist_missing,n_unknown_cat
0,6532,6.79,95561,643,12228,12.8,477,1376


,n_stage3,n_with_comment
0,8247,6449


,min_inst,max_inst,n_zero
0,0,24,2


,order_id,installments_max,price_sum
0,1a57108394169c0b47d8f876acc9ba2d,0,83.38
1,744bade1fcf9ff3f31d860ace076d422,0,45.90


,orders_all,in_window,base,master,master_with_review
0,99441,99092,96204,96204,95561


In [11]:
# [셀 11] assert 검증 → 끝에 ALL ASSERTS PASSED가 나와야 함
one = lambda q: con.execute(q).fetchone()

n_rows, n_orders = one("SELECT COUNT(*), COUNT(DISTINCT order_id) FROM master")
assert n_rows == n_orders == 96204                                          # 행 수·유일성
assert len(con.execute("DESCRIBE master").df()) == 26                       # 컬럼 수

late_master = one("SELECT SUM(late) FROM master")[0]
late_base   = one("SELECT SUM(late) FROM base")[0]
assert late_master == late_base == 6532                                     # late는 base와 같아야 함

n_review, n_no_review = one("SELECT COUNT(review_score), COUNT(*) - COUNT(review_score) FROM master")
assert (n_review, n_no_review) == (95561, 643)                              # 리뷰 있음/없음
assert one("SELECT COUNT(*) FROM master WHERE low IS NULL")[0] == n_no_review   # low NULL = 리뷰 없는 주문
assert one("SELECT SUM(low) FROM master")[0] == 12228                       # 저평점
assert one("SELECT COUNT(*), COUNT(review_comment_message) FROM master WHERE late = 0 AND low = 1") == (8247, 6449)

# dist_missing 독립 검증: item_base에서 '아이템이 전부 거리 NULL인 주문' 수와 비교
n_all_null_items = one("SELECT COUNT(*) FROM (SELECT order_id FROM item_base GROUP BY order_id HAVING COUNT(distance_km) = 0)")[0]
n_master_missing = one("SELECT SUM(dist_missing) FROM master")[0]
assert n_all_null_items == n_master_missing == 477

assert one("SELECT COUNT(*) FROM master WHERE category_en = '미상'")[0] == 1376

# 합계 대조: 대표 아이템으로 접는 과정에서 값이 새지 않았는지
p_item, f_item, n_item = one("SELECT SUM(price), SUM(freight_value), COUNT(*) FROM item_base")
p_mst,  f_mst,  n_mst  = one("SELECT SUM(price_sum), SUM(freight_sum), SUM(n_items) FROM master")
assert abs(p_item - p_mst) < 0.01 and abs(f_item - f_mst) < 0.01 and n_item == n_mst == 109873
print("가격 합 / 배송비 합 / 아이템 수:", round(p_mst, 2), round(f_mst, 2), n_mst)

assert one("SELECT MIN(installments_max), MAX(installments_max) FROM master") == (0, 24)
assert one("SELECT COUNT(*) FROM master WHERE customer_state IS NULL OR seller_state IS NULL")[0] == 0
print("ALL ASSERTS PASSED")

가격 합 / 배송비 합 / 아이템 수: 13179807.94 2191977.24 109873
ALL ASSERTS PASSED


In [12]:
# [셀 12] master.parquet 저장 → 기대: 96,204 / 96,204 / 약 15MB
import os

con.execute("COPY master TO 'data/processed/master.parquet' (FORMAT PARQUET)")
n, u = con.execute("SELECT COUNT(*), COUNT(DISTINCT order_id) FROM read_parquet('data/processed/master.parquet')").fetchone()
assert n == u == 96204
print(n, u, round(os.path.getsize("data/processed/master.parquet") / 1e6, 1), "MB")

96204 96204 15.4 MB


In [13]:
con.execute("COPY item_base TO 'data/processed/item_base.parquet' (FORMAT PARQUET)")

n, u = con.execute("""
    SELECT COUNT(*), COUNT(DISTINCT order_id) 
    FROM read_parquet('data/processed/item_base.parquet')
""").fetchone()
assert n == 109873          # item_base 행 수 (item 단위, 중복 있는 게 정상)
assert u == 96204            # 주문 수는 base와 같아야 함
print(n, u)

109873 96204


## 보충: 동점 규칙·결제 규칙 민감도 (D-05, D-06, D-12 보강)
대표 아이템을 정하는 동점 규칙(가격, 거리)과 결제 회차 `MAX` 규칙이 결과에 얼마나 영향을 주는지 확인한다. 동점 규칙을 바꿔도 D-05의 결론(절반 가까이 불일치)이 유지되는지, `seller_rep`이 동점 규칙에 의존하는 주문이 몇 건인지를 본다.

In [14]:
# [보충] 민감도 확인
import pandas as pd

# (a) D-05: 동점 규칙(가격·거리)을 뒤집었을 때 멀티셀러 일치율
#     기대(출력 순서): (ASC,ASC) 623 / 49.0 / 615,  (ASC,DESC) 592 / 46.5 / 538,  (DESC,ASC) 614 / 48.3 / 555,  (DESC,DESC) 591 / 46.5 / 584
rows = []
for price_tie in ("ASC", "DESC"):
    for dist_tie in ("ASC", "DESC"):
        r = con.execute(f"""
        WITH rp AS (SELECT order_id, order_item_id AS item_p, seller_id AS seller_p
                    FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY price DESC, order_item_id {price_tie}) AS rn
                          FROM item_base) WHERE rn = 1),
        rd AS (SELECT order_id, order_item_id AS item_d, seller_id AS seller_d
               FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY distance_km DESC NULLS LAST, order_item_id {dist_tie}) AS rn
                     FROM item_base) WHERE rn = 1),
        m AS (SELECT order_id FROM item_base GROUP BY order_id HAVING COUNT(DISTINCT seller_id) > 1)
        SELECT COUNT(*),
               CAST(SUM(CASE WHEN seller_p = seller_d THEN 1 ELSE 0 END) AS INTEGER),
               ROUND(100.0 * AVG(CASE WHEN seller_p = seller_d THEN 1 ELSE 0 END), 1),
               CAST(SUM(CASE WHEN item_p = item_d THEN 1 ELSE 0 END) AS INTEGER)
        FROM m JOIN rp USING (order_id) JOIN rd USING (order_id)
        """).fetchone()
        rows.append((price_tie, dist_tie) + r)
display(pd.DataFrame(rows, columns=["price_tie", "dist_tie", "n_multi", "same_seller", "pct_same_seller", "same_item"]))

# (b) D-12: 거리 동점 방향을 뒤집으면 seller_rep이 바뀌는 주문 → 기대: 96,204 / 143 / 0
display(con.execute("""
WITH a AS (SELECT order_id, seller_id AS s, distance_km AS d
           FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY distance_km DESC NULLS LAST, order_item_id ASC) AS rn FROM item_base) WHERE rn = 1),
b AS (SELECT order_id, seller_id AS s, distance_km AS d
      FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY distance_km DESC NULLS LAST, order_item_id DESC) AS rn FROM item_base) WHERE rn = 1)
SELECT COUNT(*) AS n_orders,
       CAST(SUM(CASE WHEN a.s <> b.s THEN 1 ELSE 0 END) AS INTEGER) AS seller_rep_differs,
       CAST(SUM(CASE WHEN a.d IS DISTINCT FROM b.d THEN 1 ELSE 0 END) AS INTEGER) AS distance_differs
FROM a JOIN b USING (order_id)
""").df())

# (c) D-06: 결제 행이 2개 이상인 base 주문, 그중 회차가 행마다 다른 주문 → 기대: 2,865 / 845
display(con.execute("""
SELECT COUNT(*) AS n_orders_multi_payment,
       CAST(SUM(CASE WHEN n_inst > 1 THEN 1 ELSE 0 END) AS INTEGER) AS n_orders_diff_installments
FROM (SELECT p.order_id, COUNT(DISTINCT p.payment_installments) AS n_inst
      FROM order_payments p JOIN base b ON b.order_id = p.order_id
      GROUP BY p.order_id HAVING COUNT(*) > 1)
""").df())

,price_tie,dist_tie,n_multi,same_seller,pct_same_seller,same_item
0,ASC,ASC,1272,623,49.0,615
1,ASC,DESC,1272,592,46.5,538
2,DESC,ASC,1272,614,48.3,555
3,DESC,DESC,1272,591,46.5,584


,n_orders,seller_rep_differs,distance_differs
0,96204,143,0


,n_orders_multi_payment,n_orders_diff_installments
0,2865,845


## 결론
- 마스터 96,204행(주문 유일), 26개 컬럼. 1단계는 96,204건, 2단계는 `low`가 NULL이 아닌 95,561건
- 멀티셀러 1,272건에서 가격 대표와 거리 대표가 같은 셀러를 가리키는 비율은 49.0%(같은 아이템 기준 48.3%). 절반 넘게 달라서 카테고리 중요도 해석 시 한계로 명시(D-05)
- 거리가 없는 주문 477건(`dist_missing = 1`), 결제 회차 0값 2건은 원본 그대로 유지
- 동점 규칙을 뒤집으면 D-05 일치율은 46.5~49.0%(결론 유지), `seller_rep`이 바뀌는 주문은 143건(0.15%), 결제 회차 `MAX`가 결과를 좌우하는 주문은 845건(base 전체의 0.88%, 결제 행이 2개 이상인 주문의 29.5%)

In [15]:
con.close()